In [ ]:
import pandas as pd
import math
from collections import Counter, defaultdict

In [ ]:
#take input text from user
text = input("Enter your text corpus: ")
tokens = text.lower().split()

Enter your text corpus: hello how are you how is your day going


In [ ]:
#vocabulary set
vocab = set(tokens)
print("\nVocabulary:", vocab)
print("Vocabulary Size:", len(vocab))


Vocabulary: {'are', 'going', 'your', 'day', 'you', 'hello', 'how', 'is'}
Vocabulary Size: 8


In [ ]:
#unigram
unigram_counts = Counter(tokens)
print("\nUnigram Counts:\n", unigram_counts)


Unigram Counts:
 Counter({'how': 2, 'hello': 1, 'are': 1, 'you': 1, 'is': 1, 'your': 1, 'day': 1, 'going': 1})


In [ ]:
bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
print("\nBigrams:\n", bigrams)

# Bigram counts
bigram_counts = defaultdict(int)
for w1, w2 in bigrams:
    bigram_counts[(w1, w2)] += 1


Bigrams:
 [('hello', 'how'), ('how', 'are'), ('are', 'you'), ('you', 'how'), ('how', 'is'), ('is', 'your'), ('your', 'day'), ('day', 'going')]


In [ ]:
def bigram_prob_raw(w1, w2):
    if unigram_counts[w1] == 0:
        return 0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

In [ ]:
V = len(vocab)

def bigram_prob_smooth(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

In [ ]:
vocab_list = list(vocab)

raw_bigram_prob = pd.DataFrame(index=vocab_list, columns=vocab_list)

for w1 in vocab_list:
    for w2 in vocab_list:
        raw_bigram_prob.loc[w1, w2] = bigram_prob_raw(w1, w2)

print("\nRaw Bigram Probability Table:\n")
print(raw_bigram_prob)


Raw Bigram Probability Table:

       are going your  day  you hello  how   is
are    0.0   0.0  0.0  0.0  1.0   0.0  0.0  0.0
going  0.0   0.0  0.0  0.0  0.0   0.0  0.0  0.0
your   0.0   0.0  0.0  1.0  0.0   0.0  0.0  0.0
day    0.0   1.0  0.0  0.0  0.0   0.0  0.0  0.0
you    0.0   0.0  0.0  0.0  0.0   0.0  1.0  0.0
hello  0.0   0.0  0.0  0.0  0.0   0.0  1.0  0.0
how    0.5   0.0  0.0  0.0  0.0   0.0  0.0  0.5
is     0.0   0.0  1.0  0.0  0.0   0.0  0.0  0.0


In [ ]:
smooth_bigram_prob = pd.DataFrame(index=vocab_list, columns=vocab_list)

for w1 in vocab_list:
    for w2 in vocab_list:
        smooth_bigram_prob.loc[w1, w2] = bigram_prob_smooth(w1, w2)

print("\nSmoothed Bigram Probability Table:\n")
print(smooth_bigram_prob)


Smoothed Bigram Probability Table:

            are     going      your       day       you     hello       how  \
are    0.111111  0.111111  0.111111  0.111111  0.222222  0.111111  0.111111   
going  0.111111  0.111111  0.111111  0.111111  0.111111  0.111111  0.111111   
your   0.111111  0.111111  0.111111  0.222222  0.111111  0.111111  0.111111   
day    0.111111  0.222222  0.111111  0.111111  0.111111  0.111111  0.111111   
you    0.111111  0.111111  0.111111  0.111111  0.111111  0.111111  0.222222   
hello  0.111111  0.111111  0.111111  0.111111  0.111111  0.111111  0.222222   
how         0.2       0.1       0.1       0.1       0.1       0.1       0.1   
is     0.111111  0.111111  0.222222  0.111111  0.111111  0.111111  0.111111   

             is  
are    0.111111  
going  0.111111  
your   0.111111  
day    0.111111  
you    0.111111  
hello  0.111111  
how         0.2  
is     0.111111  


In [ ]:
def sentence_prob(sentence, smoothing=True):
    words = sentence.lower().split()
    prob = 1

    for i in range(len(words) - 1):
        if smoothing:
            prob *= bigram_prob_smooth(words[i], words[i+1])
        else:
            prob *= bigram_prob_raw(words[i], words[i+1])

    return prob

In [ ]:
sentence = input("\nEnter a sentence to calculate probability: ")
print("Sentence Probability:", sentence_prob(sentence, smoothing=True))


Enter a sentence to calculate probability: how are you buddy how is your day going
Sentence Probability: 1.3548070246744227e-06


In [ ]:
#perplexity score
def calculate_perplexity_score(sentence, smoothing=True):
    words = sentence.lower().split()
    N = len(words)

    prob = sentence_prob(sentence, smoothing)

    if prob == 0:
        return float('inf')

    return (1 / prob) ** (1 / N)

print("Perplexity:", calculate_perplexity_score(sentence))

Perplexity: 4.487594610790195


In [ ]:
#perplexity of next word
def predict_next_word(word, smoothing=True):
    probs = {}

    for w in vocab:
        if smoothing:
            probs[w] = bigram_prob_smooth(word, w)
        else:
            probs[w] = bigram_prob_raw(word, w)

    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)

    return sorted_probs[:3]

In [ ]:
# Take word input
word = input("\nEnter a word to predict next word: ")
print("Next word predictions:", predict_next_word(word))


Enter a word to predict next word: good
Next word predictions: [('are', 0.125), ('going', 0.125), ('your', 0.125)]
